In [1]:
"""
Lab 3, Task 1 - BFS and DFS on the original A-G graph (Section 3.2) with traces.
Each node removed from the queue/stack and processed (goal included);
Skipped already-processed duplicates; 
Stopped when the goal is processed. Depth counts edges.
bfs() and dfs() are the manual's supplied code; 
the *_trace functions add frontier logging.
"""
from collections import deque

graph = {"A": ["B", "C"], "B": ["D", "E"], "C": ["F"], "D": [], "E": ["G"], "F": [], "G": []}

def bfs(start, goal):
    q = deque([(start, [start])]); seen = {start}
    while q:
        node, path = q.popleft()
        if node == goal: return path
        for n in graph[node]:
            if n not in seen:
                seen.add(n); q.append((n, path + [n]))

def dfs(start, goal):
    stack = [(start, [start])]; seen = set()
    while stack:
        node, path = stack.pop()
        if node == goal: return path
        if node in seen: continue
        seen.add(node)
        for n in reversed(graph[node]): stack.append((n, path + [n]))


def bfs_trace(g, start, goal):
    q = deque([(start, [start])]); seen = {start}; order = []; rows = []
    while q:
        node, path = q.popleft(); order.append(node)
        if node == goal:
            rows.append((node, "".join(path), [p[0] for p in q], "GOAL")); return path, order, rows
        for n in g[node]:
            if n not in seen:
                seen.add(n); q.append((n, path + [n]))
        rows.append((node, "".join(path), [p[0] for p in q], ""))
    return None, order, rows

def dfs_trace(g, start, goal):
    stack = [(start, [start])]; seen = set(); order = []; rows = []
    while stack:
        node, path = stack.pop()
        if node == goal:
            order.append(node); rows.append((node, "".join(path), [p[0] for p in stack], "GOAL")); return path, order, rows
        if node in seen: continue
        seen.add(node); order.append(node)
        for n in reversed(g[node]): stack.append((n, path + [n]))
        rows.append((node, "".join(path), [p[0] for p in stack], ""))
    return None, order, rows

def show(name, g, start, goal, fn, top_note):
    path, order, rows = fn(g, start, goal)
    print(f"--- {name} ({top_note}) ---")
    print("step | processed | path | frontier after step")
    for i, (node, p, fr, flag) in enumerate(rows, 1):
        print(i, node, "->".join(p), fr if fr else "[]", flag, sep=" | ")
    print("final path :", path)
    print("visit order:", order, f"({len(order)} nodes processed, goal included)")
    return path, order

if __name__ == "__main__":
    bp, bo = show("BFS", graph, "A", "G", bfs_trace, "queue, front = left")
    print()
    dp, do = show("DFS", graph, "A", "G", dfs_trace, "stack, top = RIGHT end; successors pushed reversed")
    print("\n--- verification against supplied code ---")
    print("supplied bfs:", bfs("A", "G"), "| trace matches:", bfs("A", "G") == bp)
    print("supplied dfs:", dfs("A", "G"), "| trace matches:", dfs("A", "G") == dp)

--- BFS (queue, front = left) ---
step | processed | path | frontier after step
1 | A | A | ['B', 'C'] | 
2 | B | A->B | ['C', 'D', 'E'] | 
3 | C | A->C | ['D', 'E', 'F'] | 
4 | D | A->B->D | ['E', 'F'] | 
5 | E | A->B->E | ['F', 'G'] | 
6 | F | A->C->F | ['G'] | 
7 | G | A->B->E->G | [] | GOAL
final path : ['A', 'B', 'E', 'G']
visit order: ['A', 'B', 'C', 'D', 'E', 'F', 'G'] (7 nodes processed, goal included)

--- DFS (stack, top = RIGHT end; successors pushed reversed) ---
step | processed | path | frontier after step
1 | A | A | ['C', 'B'] | 
2 | B | A->B | ['C', 'E', 'D'] | 
3 | D | A->B->D | ['C', 'E'] | 
4 | E | A->B->E | ['C', 'G'] | 
5 | G | A->B->E->G | ['C'] | GOAL
final path : ['A', 'B', 'E', 'G']
visit order: ['A', 'B', 'D', 'E', 'G'] (5 nodes processed, goal included)

--- verification against supplied code ---
supplied bfs: ['A', 'B', 'E', 'G'] | trace matches: True
supplied dfs: ['A', 'B', 'E', 'G'] | trace matches: True


In [2]:
"""
Lab 3, Task 2 - DLS and IDDFS on the original A-G graph, limits 1-4.
Root has depth 0. Cycle check is PATH-LOCAL (a node already on the current path is
skipped); 
No global visited set is carried across depth limits.
Status: FOUND (goal reached), CUTOFF (a deeper unexplored continuation exists),
FAILURE (explored branch exhausted with no goal).
"""
class Counter:
    def __init__(self): self.expanded = 0

def dls(g, node, goal, limit, path=None, counter=None):
    """Recursive depth-limited search. Returns (status, path)."""
    path = path or [node]
    counter = counter or Counter()
    counter.expanded += 1
    if node == goal:
        return "FOUND", path
    if len(path) - 1 == limit:                       # at the depth limit
        unexplored = [n for n in g[node] if n not in path]
        return ("CUTOFF", None) if unexplored else ("FAILURE", None)
    cutoff = False
    for n in g[node]:
        if n in path:                                # path-local cycle check
            continue
        status, res = dls(g, n, goal, limit, path + [n], counter)
        if status == "FOUND": return status, res
        if status == "CUTOFF": cutoff = True
    return ("CUTOFF", None) if cutoff else ("FAILURE", None)

def dls_counted(g, start, goal, limit):
    c = Counter(); s, p = dls(g, start, goal, limit, [start], c); return s, p, c.expanded

def iddfs(g, start, goal, max_depth):
    """Repeat DLS with limits 0..max_depth (fresh search each time)."""
    attempted, total = [], 0
    last = "FAILURE"
    for limit in range(0, max_depth + 1):
        s, p, n = dls_counted(g, start, goal, limit)
        attempted.append((limit, s, n)); total += n
        if s == "FOUND": return "FOUND", p, attempted, total
        last = s if s == "CUTOFF" else last
        if s == "FAILURE": return "FAILURE", None, attempted, total   # whole graph exhausted
    return last, None, attempted, total

if __name__ == "__main__":
    print("=== DLS, start A, goal G ===")
    print("limit | status | path | nodes expanded")
    for lim in (1, 2, 3, 4):
        s, p, n = dls_counted(graph, "A", "G", lim)
        print(lim, s, "->".join(p) if p else None, n, sep=" | ")
    print("\n=== IDDFS (limits 0..max, stops at first FOUND) ===")
    for mx in (1, 2, 3, 4):
        s, p, att, total = iddfs(graph, "A", "G", mx)
        print(f"max depth {mx}: {s} | path {'->'.join(p) if p else None} | "
              f"limits attempted {[a[0] for a in att]} | per-limit expansions {[a[2] for a in att]} | total {total}")

=== DLS, start A, goal G ===
limit | status | path | nodes expanded
1 | CUTOFF | None | 3
2 | CUTOFF | None | 6
3 | FOUND | A->B->E->G | 5
4 | FOUND | A->B->E->G | 5

=== IDDFS (limits 0..max, stops at first FOUND) ===
max depth 1: CUTOFF | path None | limits attempted [0, 1] | per-limit expansions [1, 3] | total 4
max depth 2: CUTOFF | path None | limits attempted [0, 1, 2] | per-limit expansions [1, 3, 6] | total 10
max depth 3: FOUND | path A->B->E->G | limits attempted [0, 1, 2, 3] | per-limit expansions [1, 3, 6, 5] | total 15
max depth 4: FOUND | path A->B->E->G | limits attempted [0, 1, 2, 3] | per-limit expansions [1, 3, 6, 5] | total 15


In [3]:
"""Lab 3, Task 3 - Uniform-Cost Search on the weighted extension (Section 3.7).
Extended topology = original A-G graph PLUS the documented extra corridor C -> G.
All weights are nonnegative travel-cost units (SYNTHETIC lab assumptions chosen to
reproduce the manual's stated results: A-C-G = 12, A-B-E-G = 3).
Tie-breaking: equal cost -> earlier insertion into the frontier is processed first.
"""
import heapq
from collections import deque

# successor order = tie-breaking order; C -> G is the added corridor
weights = {
    "A": [("B", 1), ("C", 2)],
    "B": [("D", 4), ("E", 1)],
    "C": [("F", 3), ("G", 10)],      # C->G ADDED for the weighted experiment
    "D": [], "E": [("G", 1)], "F": [], "G": [],
}
topology = {k: [n for n, _ in v] for k, v in weights.items()}   # same graph, weights dropped

def ucs(start, goal):
    counter = 0
    frontier = [(0, counter, start, [start])]
    best = {start: 0}; processed = set(); order = []; rows = []
    while frontier:
        cost, _, node, path = heapq.heappop(frontier)
        if node in processed: continue                # stale (worse) duplicate entry
        processed.add(node); order.append((node, cost))
        if node == goal:
            rows.append((node, cost, path, sorted((c, n) for c, _, n, _ in frontier), "GOAL"))
            return path, cost, order, rows
        for n, w in weights[node]:
            new = cost + w
            if n not in processed and new < best.get(n, float("inf")):
                best[n] = new; counter += 1
                heapq.heappush(frontier, (new, counter, n, path + [n]))
        rows.append((node, cost, path, sorted((c, n) for c, _, n, _ in frontier), ""))
    return None, None, order, rows

def path_cost(path):
    """INDEPENDENT cost check: re-sum edge weights from the table (not from ucs)."""
    total = 0
    for a, b in zip(path, path[1:]):
        total += dict(weights[a])[b]
    return total

def bfs_topology(start, goal):
    q = deque([(start, [start])]); seen = {start}; order = []
    while q:
        node, path = q.popleft(); order.append(node)
        if node == goal: return path, order
        for n in topology[node]:
            if n not in seen: seen.add(n); q.append((n, path + [n]))
    return None, order

if __name__ == "__main__":
    print("Weighted edges (added corridor marked *):")
    for a, lst in weights.items():
        for b, w in lst:
            print(f"  {a} -> {b} : {w}{'  *added' if (a, b) == ('C', 'G') else ''}")
    path, cost, order, rows = ucs("A", "G")
    print("\n=== UCS trace ===")
    print("step | processed | g(n) | path | frontier after step (cost,node)")
    for i, (node, c, p, fr, flag) in enumerate(rows, 1):
        print(i, node, c, "->".join(p), fr, flag, sep=" | ")
    print("UCS path :", path, "| hops:", len(path) - 1, "| reported cost:", cost)
    print("UCS processing order:", [n for n, _ in order], f"({len(order)} processed, goal included)")
    print("independent cost sum :", path_cost(path))
    bp, bo = bfs_topology("A", "G")
    print("\n=== BFS on the same weighted topology ===")
    print("BFS path :", bp, "| hops:", len(bp) - 1, "| independent cost:", path_cost(bp))
    print("BFS processing order:", bo)
    print("\n=== Comparison ===")
    print(f"BFS: hops={len(bp)-1}, cost={path_cost(bp)}   UCS: hops={len(path)-1}, cost={path_cost(path)}")

Weighted edges (added corridor marked *):
  A -> B : 1
  A -> C : 2
  B -> D : 4
  B -> E : 1
  C -> F : 3
  C -> G : 10  *added
  E -> G : 1

=== UCS trace ===
step | processed | g(n) | path | frontier after step (cost,node)
1 | A | 0 | A | [(1, 'B'), (2, 'C')] | 
2 | B | 1 | A->B | [(2, 'C'), (2, 'E'), (5, 'D')] | 
3 | C | 2 | A->C | [(2, 'E'), (5, 'D'), (5, 'F'), (12, 'G')] | 
4 | E | 2 | A->B->E | [(3, 'G'), (5, 'D'), (5, 'F'), (12, 'G')] | 
5 | G | 3 | A->B->E->G | [(5, 'D'), (5, 'F'), (12, 'G')] | GOAL
UCS path : ['A', 'B', 'E', 'G'] | hops: 3 | reported cost: 3
UCS processing order: ['A', 'B', 'C', 'E', 'G'] (5 processed, goal included)
independent cost sum : 3

=== BFS on the same weighted topology ===
BFS path : ['A', 'C', 'G'] | hops: 2 | independent cost: 12
BFS processing order: ['A', 'B', 'C', 'D', 'E', 'F', 'G']

=== Comparison ===
BFS: hops=2, cost=12   UCS: hops=3, cost=3


In [4]:
"""Lab 3 - edge-case tests (Section 3.8): 
start==goal, unreachable goal, cycle.
(Runs in this notebook after the cells above.)
"""
# Single-notebook version: bfs_trace, dfs_trace, dls_counted, iddfs, ucs and graph
# are defined in the cells above, so no imports or module prefixes are needed.
def bfs_g(g, s, t):  return bfs_trace(g, s, t)[0]
def dfs_g(g, s, t):  return dfs_trace(g, s, t)[0]

print("=== Test 1: start == goal (A -> A) ===")
print("BFS", bfs_g(graph, "A", "A"), "| DFS", dfs_g(graph, "A", "A"),
      "| DLS(limit 0)", dls_counted(graph, "A", "A", 0)[:2], "| UCS", ucs("A", "A")[:2])

print("\n=== Test 2: unreachable goal (A -> Z, Z not in graph's reachable set) ===")
g2 = dict(graph); g2["Z"] = []            # Z exists but has no incoming edge
print("BFS", bfs_g(g2, "A", "Z"), "| DFS", dfs_g(g2, "A", "Z"))
for lim in (2, 10):
    s, p, n = dls_counted(g2, "A", "Z", lim)
    print(f"DLS limit {lim}: {s} (expanded {n})")
s, p, att, tot = iddfs(g2, "A", "Z", 10)
print("IDDFS max 10:", s, "| limits attempted", [a[0] for a in att], "| total expansions", tot)

print("\n=== Test 3: graph with a cycle (A->B->C->A, C->D) ===")
cyc = {"A": ["B"], "B": ["C"], "C": ["A", "D"], "D": []}
print("BFS A->D :", bfs_g(cyc, "A", "D"))
print("DFS A->D :", dfs_g(cyc, "A", "D"))
print("DLS limit 3:", dls_counted(cyc, "A", "D", 3)[:2])
print("BFS A->Q (unreachable) terminates:", bfs_g({**cyc, "Q": []}, "A", "Q"))
print("DLS limit 10 A->Q:", dls_counted({**cyc, "Q": []}, "A", "Q", 10)[:2])

print("\n=== Test 4: bounded result is not proof (limit 2 on A->G) ===")
print("DLS limit 2:", dls_counted(graph, "A", "G", 2)[:2], "-> CUTOFF only; G IS reachable at depth 3")

=== Test 1: start == goal (A -> A) ===
BFS ['A'] | DFS ['A'] | DLS(limit 0) ('FOUND', ['A']) | UCS (['A'], 0)

=== Test 2: unreachable goal (A -> Z, Z not in graph's reachable set) ===
BFS None | DFS None
DLS limit 2: CUTOFF (expanded 6)
DLS limit 10: FAILURE (expanded 7)
IDDFS max 10: FAILURE | limits attempted [0, 1, 2, 3] | total expansions 17

=== Test 3: graph with a cycle (A->B->C->A, C->D) ===
BFS A->D : ['A', 'B', 'C', 'D']
DFS A->D : ['A', 'B', 'C', 'D']
DLS limit 3: ('FOUND', ['A', 'B', 'C', 'D'])
BFS A->Q (unreachable) terminates: None
DLS limit 10 A->Q: ('FAILURE', None)

=== Test 4: bounded result is not proof (limit 2 on A->G) ===
DLS limit 2: ('CUTOFF', None) -> CUTOFF only; G IS reachable at depth 3
